# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd, numpy as np
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

base = df[df["clicks_prev_30d"] > 0].copy()
y = (base["clicks_last_30d"] < base["clicks_prev_30d"]).astype(int)
groups = base["client_id"]

cat_cols = ["content_type", "main_intent", "competition_level", "provider_used", "model_used"]
num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
            "content_age_days", "days_since_last_update",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
X = base[num_cols + cat_cols].copy()
X["chars_per_word"] = base["char_count"] / base["word_count"].replace(0, np.nan)
X["prev_ctr"] = base["clicks_prev_30d"] / base["impressions_prev_30d"].replace(0, np.nan)
X["log_clicks_prev"] = np.log1p(base["clicks_prev_30d"])
X["log_impr_prev"] = np.log1p(base["impressions_prev_30d"])
X = X.replace([np.inf, -np.inf], np.nan)
for c in list(X.select_dtypes("number").columns):
    if X[c].isna().any():
        X[c + "_was_missing"] = X[c].isna().astype(int)
        X[c] = X[c].fillna(X[c].median())
X[cat_cols] = X[cat_cols].fillna("unknown")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype(float)
print(X.shape, "| positive rate:", y.mean().round(3))

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
(11759, 35) | positive rate: 0.579


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
share = groups.value_counts(normalize=True)
print("clients:", len(share), "| largest client share:", share.iloc[0].round(3),
      "| top 5 share:", share.head(5).sum().round(3))
print("client decline-rate spread:\n", base.assign(y=y).groupby("client_id")["y"].mean().describe().round(3))

clients: 29 | largest client share: 0.307 | top 5 share: 0.754
client decline-rate spread:
 count    29.000
mean      0.649
std       0.171
min       0.300
25%       0.553
50%       0.622
75%       0.770
max       1.000
Name: y, dtype: float64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def cv_scores(splitter, **kw):
    out = []
    for tr, te in splitter.split(X, y, **kw):
        m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=0)
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        out.append([roc_auc_score(y.iloc[te], p), average_precision_score(y.iloc[te], p)])
    return np.array(out)

before = cv_scores(StratifiedKFold(5, shuffle=True, random_state=0))
after = cv_scores(GroupKFold(5), groups=groups)

res = pd.DataFrame({
    "split": ["BEFORE: random", "AFTER: grouped by client"],
    "AUC mean": [before[:, 0].mean(), after[:, 0].mean()],
    "AUC std": [before[:, 0].std(), after[:, 0].std()],
    "AvgPrec mean": [before[:, 1].mean(), after[:, 1].mean()],
}).round(3)
print(res.to_string(index=False))
print("\nAUC drop from honest split:", round(before[:, 0].mean() - after[:, 0].mean(), 3))
print("base rate (random AvgPrec):", y.mean().round(3))

                   split  AUC mean  AUC std  AvgPrec mean
          BEFORE: random     0.698    0.009         0.744
AFTER: grouped by client     0.646    0.007         0.693

AUC drop from honest split: 0.052
base rate (random AvgPrec): 0.579


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
# (a) Single-feature check: nothing legitimate should be near-perfect
rows = []
for c in X.columns:
    try:
        a = roc_auc_score(y, X[c]); rows.append((c, max(a, 1 - a)))
    except Exception:
        pass
single = pd.DataFrame(rows, columns=["feature", "auc"]).sort_values("auc", ascending=False)
print(single.head(8).round(3).to_string(index=False))

# (b) Banned-pattern check on the raw columns used
used = num_cols + cat_cols
banned = ["_90d", "last_30d", "tier", "trend_", "ctr", "engagement", "scroll_rate", "ai_traffic", "days_with_"]
print("\nbanned patterns in raw inputs:", [c for c in used if any(b in c for b in banned)] or "none")
print("engineered from banned columns:", "none (prev_ctr uses only *_prev_30d)")

# (c) Positive control: add the future window and watch AUC jump (proves the test can detect a leak)
Xleak = X.copy()
Xleak["LEAK_clicks_last_30d"] = base["clicks_last_30d"].values
m = HistGradientBoostingClassifier(max_depth=4, max_iter=100, random_state=0)
tr, te = next(GroupKFold(5).split(Xleak, y, groups))
for name, XX in [("final features", X), ("with leaked column", Xleak)]:
    m.fit(XX.iloc[tr], y.iloc[tr])
    print(f"{name}: AUC = {roc_auc_score(y.iloc[te], m.predict_proba(XX.iloc[te])[:, 1]):.3f}")

# (d) Can features identify the client? (client fingerprinting)
print("\nduplicate feature rows:", X.duplicated().sum())

                          feature   auc
                         prev_ctr 0.673
                  clicks_prev_30d 0.561
                  log_clicks_prev 0.561
model_used_gemini-3-flash-preview 0.555
                    log_impr_prev 0.555
             impressions_prev_30d 0.555
                   chars_per_word 0.550
                       word_count 0.536

banned patterns in raw inputs: none
engineered from banned columns: none (prev_ctr uses only *_prev_30d)
final features: AUC = 0.636
with leaked column: AUC = 0.999

duplicate feature rows: 3


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
print("rows:", len(base), "| clients:", groups.nunique())
print("grouped-split AUC: mean", round(after[:, 0].mean(), 3), "| std", round(after[:, 0].std(), 3))
print("random-split AUC (before):", round(before[:, 0].mean(), 3))
print("decline base rate:", y.mean().round(3))

rows: 11759 | clients: 29
grouped-split AUC: mean 0.646 | std 0.007
random-split AUC (before): 0.698
decline base rate: 0.579


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.